In [ ]:
import pandas as pd

df=pd.read_csv('https://raw.githubusercontent.com/pykwon/python/master/testdata_utf8/articlesapril.csv')
print(df.head())
print(df.shape)
print(df.columns)
print(df['headline'].head())
print(df['headline'].isnull().values.any())
print(df['headline'].values)
print('---')
#headline 열에서 모든 신문 기사의 제목을 뽑아서 하나의 리스트로 저장해 보도록 하겠다.
headline = [] # 리스트 선언
headline.extend(list(df.headline.values)) # 헤드라인의 값들을 리스트로 저장
print(headline[:5]) # 상위 5개만 출력

# 그런데 4번째, 5번째 샘플에 Unknown 값이 들어가 있다. headline 전체에 걸쳐서 Unknown 값을 가진 샘플이 있을 것으로 추정. 비록 Null 값은 아니지만 지금 하고자 하는 실습에 도움이 되지 않는 노이즈 데이터이므로 제거해 줄 필요가 있다. 제거하기 전에 현재 샘플의 개수를 확인해보고 제거 전,후 샘플의 개수를 비교해보겠다.
print(len(headline))  # 현재 샘플의 개수
# 노이즈 데이터를 제거하기 전 데이터의 개수는 1,324. 즉, 신문 기사의 제목이 총 1,324개다.

headline = [n for n in headline if n != "Unknown"] # Unknown 값을 가진 샘플 제거
print(len(headline))  # 제거 후 샘플의 개수   1214
# Unknown 값을 가진 샘플을 제거한 후 샘플의 개수는 1,214. 110개의 샘플이 제거되었는데, 기존에 출력했던 5개의 샘플을 출력해 보겠다.
print(headline[:5])

# 데이터 전처리를 수행. 여기서 선택한 전처리는 구두점 제거와 단어의 소문자화이다.
from string import punctuation
def repreprocessing(s):
    s=s.encode("utf8").decode("ascii",'ignore')
    return ''.join(c for c in s if c not in punctuation).lower()     # 구두점 제거와 소문자화

text = [repreprocessing(x) for x in headline]
print(text[:5])
# 기존의 출력과 비교하면 모든 단어들이 소문자화되었으며 N.F.L.이나 Cheerleaders’ 등과 같이 기존에 구두점이 붙어있던 단어들에서 구두점이 제거되었다.

# 이제 단어 집합(vocabulary)을 만든다.
from tensorflow.keras.preprocessing.text import Tokenizer

tok = Tokenizer()
tok.fit_on_texts(text)   # 토큰화
vocab_size = len(tok.word_index) + 1
print('단어 집합의 크기 : %d' % vocab_size)    # 3494

# 이제 훈련 데이터의 형태로 만들어본다.
# 정수 인코딩을 수행하는 동시에 하나의 문장을 여러 줄로 분해할 것이다.
sequences = list()
for line in text:
    enc = tok.texts_to_sequences([line])[0]  # 각 샘플에 대한 정수 인코딩
    for i in range(1, len(enc)):
        se = enc[:i+1]
        sequences.append(se)
# print(sequences)
print(sequences[:11])

print('dict items:', list(tok.word_index.items())[:5])
index_to_word={}
for key, value in tok.word_index.items():  # 인덱스를 단어로 바꾸기 위해 index_to_word를 생성
    index_to_word[value] = key

print(index_to_word[582])  # offer
# 582이라는 인덱스를 가진 단어는 본래 offer라는 단어였다. 원한다면 다른 숫자로도 시도해보라.
# 이제 y데이터를 분리하기 전에 전체 샘플의 길이를 동일하게 만드는 패딩 작업을 수행한다.
# 패딩 작업을 수행하기 전에 가장 긴 샘플의 길이를 확인한다.

max_len=max(len(l) for l in sequences)
print(max_len)   # 24
# 신문 기사의 본문이 아닌 제목이므로 샘플 길이가 길지는 않다. 가장 긴 샘플의 길이인 24로 모든 샘플의 길이를 맞추겠다.

from keras.preprocessing.sequence import pad_sequences
sequences = pad_sequences(sequences, maxlen=max_len, padding='pre')
print(sequences[:3])

import numpy as np
sequences = np.array(sequences)
X = sequences[:,:-1]   # feature
y = sequences[:,-1]    # label
print(X[:3])
# 훈련 데이터 X에서 3개의 샘플만 출력해보았는데, 맨 우측에 있던 정수값 269, 371, 1115가 사라진 것을 볼 수 있다. 뿐만 아니라, 각 샘플의 길이가 24에서 23으로 줄었다.
print(y[:3])     # 레이블   [ 269  371 1115]
# 훈련 데이터 y 중 3개의 샘플만 출력해 보았는데, 기존 훈련 데이터에서 맨 우측에 있던 정수들이 별도로 저장되었다.

from keras.utils import to_categorical
y = to_categorical(y, num_classes=vocab_size)

# 레이블 데이터 y에 대해서 원-핫 인코딩을 수행하였다. 이제 모델을 설계.
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.models import Sequential

model = Sequential()
model.add(Embedding(vocab_size, 10, input_length=max_len-1))
# y 데이터를 분리하였으므로 이제 X 데이터의 길이는 기존 데이터의 길이 - 1
model.add(LSTM(128, activation='tanh'))
model.add(Dense(32, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X, y, epochs=100, verbose=2)   # 속도 느림
print(model.evaluate(X, y))  # [0.23405415393382514, 0.92695117]

# 문장을 생성하는 함수 sentence_generation을 만들어서 출력해본다.
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def sentence_generation(model, t, current_word, n):
    init_word = current_word
    generated = []

    index_to_word = getattr(t, "index_word", None)
    if not index_to_word:
        index_to_word = {idx: w for w, idx in t.word_index.items()}

    for _ in range(n):
        enc = t.texts_to_sequences([current_word])[0]
        if len(enc) == 0:          # 시작 단어가 사전에 없으면 중단
            break

        enc = pad_sequences([enc], maxlen=max_len - 1, padding='pre')

        pred = model.predict(enc, verbose=0)
        idx = int(np.argmax(pred, axis=-1)[0])   # 정수 인덱스 추출

        if idx == 0 or idx not in index_to_word: # 패딩/미정 단어는 중단
            break

        word = index_to_word[idx]
        generated.append(word)
        current_word = current_word + ' ' + word

    return (init_word + ' ' + ' '.join(generated)).strip()

print(sentence_generation(model, tok, 'i', 10))   # 임의의 단어 'i'에 대해서 10개 단어를 추가 생성
#i disapprove of school vouchers can i still apply for them
print(sentence_generation(model, tok, 'how', 50))
# how to make facebook more accountable will so your neighbor chasing
